In [18]:
import xarray as xr
import numpy as np
from pathlib import Path

In [19]:
ds = xr.open_dataset("raw data/TC_snapshot/TC_1.nc")

In [23]:
# =====================================================
# STREAMING GLOBAL HOURLY TC WIND SPEED FIELD (2010+)
# =====================================================

import numpy as np
import xarray as xr
from pathlib import Path
from scipy.interpolate import griddata
from tqdm import tqdm
import netCDF4 as nc

# -----------------------------
# PATH TO TC SNAPSHOTS
# -----------------------------
tc_dir = Path("raw data/TC_snapshot")
tc_files = sorted(tc_dir.glob("TC_*.nc"))

print(f"Found {len(tc_files)} TC files")
assert len(tc_files) > 0

# -----------------------------
# TIME FILTER
# -----------------------------
start_time = np.datetime64("2010-01-01T00:00")

# -----------------------------
# GLOBAL GRID
# -----------------------------
lat_global = np.arange(-90.0, 90.25, 0.25)
lon_global = np.arange(0.0, 360.0, 0.25)

nlat = len(lat_global)
nlon = len(lon_global)

# -----------------------------
# COLLECT UNIQUE TIMES (2010+)
# -----------------------------
all_times = []

for f in tc_files:
    ds = xr.open_dataset(f)
    if "dimdt" not in ds.dims:
        continue

    times = ds.Time.values
    times = times[times >= start_time]   # 🔑 FILTER HERE

    if len(times) > 0:
        all_times.append(times)

assert len(all_times) > 0, "No TC data found after 2010"

all_times = np.unique(np.concatenate(all_times))
all_times.sort()

ntime = len(all_times)
time_index = {t: i for i, t in enumerate(all_times)}

print(f"Total unique hourly timesteps (2010+): {ntime}")

# -----------------------------
# CREATE NETCDF FILE (ON DISK)
# -----------------------------
ncfile = nc.Dataset(
    "TC_global_wind_speed_hourly_2010_2022.nc",
    "w",
    format="NETCDF4"
)

ncfile.createDimension("time", ntime)
ncfile.createDimension("lat", nlat)
ncfile.createDimension("lon", nlon)

time_var = ncfile.createVariable("time", "f8", ("time",))
lat_var  = ncfile.createVariable("lat",  "f4", ("lat",))
lon_var  = ncfile.createVariable("lon",  "f4", ("lon",))

wind_var = ncfile.createVariable(
    "tc_wind_speed",
    "f4",
    ("time", "lat", "lon"),
    zlib=True,
    complevel=4,
    fill_value=np.nan
)

lat_var[:] = lat_global
lon_var[:] = lon_global
time_var[:] = xr.DataArray(all_times).astype("datetime64[h]").values.astype("float64")

# -----------------------------
# STREAMING TC PROCESSING (2010+)
# -----------------------------
for f in tqdm(tc_files, desc="Processing TCs"):
    ds = xr.open_dataset(f)

    if "dimdt" not in ds.dims:
        continue

    for i in range(ds.sizes["dimdt"]):
        time_val = ds.Time.isel(dimdt=i).values

        # 🔑 SKIP EARLY YEARS
        if time_val < start_time:
            continue

        t_idx = time_index[time_val]

        ws = np.sqrt(
            ds.u10.isel(dimdt=i)**2 + ds.v10.isel(dimdt=i)**2
        ).values

        if np.all(np.isnan(ws)):
            continue

        lon = ds.lon.isel(dimdt=i).values
        lat = ds.lat.isel(dimdt=i).values
        lon2d, lat2d = np.meshgrid(lon, lat)

        points = np.column_stack([lon2d.ravel(), lat2d.ravel()])
        values = ws.ravel()

        # bounding box
        lon_min, lon_max = lon.min() - 1.0, lon.max() + 1.0
        lat_min, lat_max = lat.min() - 1.0, lat.max() + 1.0

        lon_mask = (lon_global >= lon_min) & (lon_global <= lon_max)
        lat_mask = (lat_global >= lat_min) & (lat_global <= lat_max)

        lon_sub = lon_global[lon_mask]
        lat_sub = lat_global[lat_mask]
        lon2s, lat2s = np.meshgrid(lon_sub, lat_sub)

        grid_ws = griddata(
            points,
            values,
            (lon2s, lat2s),
            method="nearest"
        )

        existing = wind_var[t_idx, lat_mask, lon_mask]

        wind_var[t_idx, lat_mask, lon_mask] = np.nanmax(
            np.stack([existing, grid_ws]), axis=0
        )

# -----------------------------
# FINALIZE
# -----------------------------
ncfile.close()
print("Saved: TC_global_wind_speed_hourly_2010_2022.nc")


Found 2932 TC files
Total unique hourly timesteps (2010+): 89937


Processing TCs:   0%|          | 0/2932 [00:00<?, ?it/s]C:\Users\emile\AppData\Local\Temp\ipykernel_22260\1201124508.py:100: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  for i in range(ds.dims["dimdt"]):
C:\Users\emile\AppData\Local\Temp\ipykernel_22260\1201124508.py:100: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  for i in range(ds.dims["dimdt"]):
C:\Users\emile\AppData\Local\Temp\ipykernel_22260\1201124508.py:100: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To acces

KeyboardInterrupt: 